## Problem 1

Using the model from class, replace MobileNetV2 with ResNet50.

Evaluate:

- Training time
- Accuracy
- Model size


In [1]:
import tensorflow as tf
import time

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train[:5000]
y_train = y_train[:5000]

x_test = x_test[:1000]
y_test = y_test[:1000]

x_train = tf.image.resize(x_train, (96, 96))
x_test = tf.image.resize(x_test, (96, 96))

x_train = tf.keras.applications.resnet50.preprocess_input(x_train)
x_test = tf.keras.applications.resnet50.preprocess_input(x_test)

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(96, 96, 3))

base_model.trainable = False
resnet_model = Sequential([base_model, GlobalAveragePooling2D(), Dense(10, activation='softmax')])
resnet_model.summary()

start_time = time.time()
history = resnet_model.fit(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))
training_time = time.time() - start_time

test_loss, test_accuracy = resnet_model.evaluate(x_test, y_test, verbose=0)
parameter_count = resnet_model.count_params()

model_size = resnet_model.count_params() * 4 / (1024 ** 2)  

print(f"Training time: {training_time:.2f} seconds")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Number of parameters: {parameter_count}")
print(f"Model size: {model_size:.2f} MB")


2026-07-31 23:31:09.247521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1535s 9us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 3, 3, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

ValueError: You must call `compile()` before using the model.


## Problem 2

Create a transfer learning model using the TensorFlow Flowers dataset and classify five flower species.


In [2]:
import pathlib
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    RandomFlip,
    RandomRotation,
    Rescaling,
    GlobalAveragePooling2D,
    Dropout,
    Dense
)

dataset_url = (
    "https://storage.googleapis.com/"
    "download.tensorflow.org/example_images/flower_photos.tgz"
)

data_directory = tf.keras.utils.get_file(
    "flower_photos",
    origin=dataset_url,
    untar=True
)

data_directory = pathlib.Path(data_directory)

image_size = (160, 160)
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    data_directory,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    data_directory,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

class_names = train_dataset.class_names
print(f"Flower classes: {class_names}")

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)

validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

data_augmentation = Sequential(RandomFlip("horizontal"), RandomRotation(0.2))

base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights="imagenet")
base_model.trainable = False

flower_model = Sequential([
    data_augmentation,
    tf.keras.layers.Rescaling(1./127.5, offset=-1),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(5, activation="softmax")
])

flower_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']    
)

history = flower_model.fit(
    train_dataset,
    epochs=5,
    validation_data=validation_dataset
)

validation_loss, validation_accuracy = flower_model.evaluate(validation_dataset, verbose=0)

print(f"Validation accuracy: {validation_accuracy:.4f}")

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
Found 3670 files belonging to 1 classes.
Using 2936 files for training.
Found 3670 files belonging to 1 classes.
Using 734 files for validation.
Flower classes: ['flower_photos']


ValueError: Expected `trainable` to be a boolean. Received: trainable=<RandomRotation name=random_rotation, built=False> (of type <class 'keras.src.layers.preprocessing.image_preprocessing.random_rotation.RandomRotation'>)